# 03_silver
Pasa las respuestas JSON de `workspace.inferiores.bronze_player_raw` a tablas silver, una por tema. Cada tabla se reescribe completa (`overwrite`) en `workspace.inferiores`.

Todas las celdas parten de la misma regla: para cada jugador toman la respuesta **más reciente** (`fecha` más nueva) de la llamada correspondiente con estado `hecho`, y descomponen el JSON en columnas.

`orden` es la posición del elemento dentro de la lista original de la respuesta, empezando en 1. En `silver_national_team_career` se vuelve a numerar después de quitar duplicados.

## Contenido

| Tabla | Llamada de origen | Una fila por |
|---|---|---|
| `silver_players` | `profile` | jugador |
| `silver_player_citizenship` | `profile` (campo `citizenship`) | jugador y nacionalidad |
| `silver_market_value_history` | `market_value` | jugador y punto de la serie de valor de mercado |
| `silver_national_team_career` | `national_career` | jugador y selección, sin duplicados exactos |
| `silver_achievements` | `achievements` | jugador, título y detalle del título |
| `silver_player_stats` | `stats` | jugador, competición, temporada y club |
| `silver_transfers` | `transfers` | jugador y traspaso |
| `silver_youth_clubs` | `transfers` (campo `youthClubs`) | jugador y club juvenil |

Después de escribir cada tabla, la celda muestra una consulta de control con conteos de filas, jugadores y claves únicas.

## Control del ciclo
`%run ./ciclo` carga las funciones del notebook `ciclo`. La celda siguiente llama a `motivo_no_corre("publicar")`: si el ciclo no está listo para publicar (todavía faltan jugadores o ya se publicó), termina con `SALTEADO: <motivo>` y no escribe nada. Este notebook no marca `publicado_en`.

In [0]:
%run ./ciclo

In [0]:
motivo_salida = motivo_no_corre("publicar")
if motivo_salida:
    dbutils.notebook.exit("SALTEADO: " + motivo_salida)

## silver_players
Una fila por jugador, a partir del último `profile`.

| Columnas | Contenido |
|---|---|
| `name`, `name_in_home_country`, `full_name` | Nombres |
| `date_of_birth`, `date_of_death`, `birth_city`, `birth_country` | Nacimiento y fallecimiento |
| `height_cm`, `foot`, `position_main`, `position_group`, `shirt_number` | Datos físicos y de posición. `shirt_number` se guarda sin el `#` |
| `is_retired`, `retired_since` | Retiro |
| `club_id`, `club_name`, `club_joined`, `contract_expires`, `agent_name` | Club actual y contrato |
| `url`, `image_url`, `updated_at` | Enlaces y fecha de actualización de la respuesta |

Control: cantidad de filas y de jugadores distintos.

In [0]:
# EN LIMPIO: silver_players (escribe workspace.inferiores.silver_players)
q = """
WITH ult AS (
  SELECT player_id, body,
         ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY fecha DESC) AS rn
  FROM workspace.inferiores.bronze_player_raw
  WHERE llamada = 'profile' AND estado = 'hecho'
)
SELECT
  player_id,
  get_json_object(body, '$.name')                                   AS name,
  get_json_object(body, '$.nameInHomeCountry')                      AS name_in_home_country,
  get_json_object(body, '$.fullName')                               AS full_name,
  to_date(get_json_object(body, '$.dateOfBirth'))                   AS date_of_birth,
  to_date(get_json_object(body, '$.dateOfDeath'))                   AS date_of_death,
  get_json_object(body, '$.placeOfBirth.city')                      AS birth_city,
  get_json_object(body, '$.placeOfBirth.country')                   AS birth_country,
  CAST(get_json_object(body, '$.height') AS INT)                    AS height_cm,
  get_json_object(body, '$.foot')                                   AS foot,
  get_json_object(body, '$.position.main')                          AS position_main,
  get_json_object(body, '$.position.group')                         AS position_group,
  CAST(regexp_replace(get_json_object(body, '$.shirtNumber'), '#', '') AS INT) AS shirt_number,
  CAST(get_json_object(body, '$.isRetired') AS BOOLEAN)             AS is_retired,
  to_date(get_json_object(body, '$.retiredSince'))                  AS retired_since,
  get_json_object(body, '$.club.id')                                AS club_id,
  get_json_object(body, '$.club.name')                              AS club_name,
  to_date(get_json_object(body, '$.club.joined'))                   AS club_joined,
  to_date(get_json_object(body, '$.club.contractExpires'))          AS contract_expires,
  get_json_object(body, '$.agent.name')                             AS agent_name,
  get_json_object(body, '$.url')                                    AS url,
  get_json_object(body, '$.imageUrl')                               AS image_url,
  to_timestamp(get_json_object(body, '$.updatedAt'))                AS updated_at
FROM ult WHERE rn = 1
"""
(spark.sql(q).write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.inferiores.silver_players"))

display(spark.sql("""SELECT COUNT(*) AS filas, COUNT(DISTINCT player_id) AS jugadores
                     FROM workspace.inferiores.silver_players"""))

## silver_player_citizenship
Una fila por jugador y nacionalidad, a partir de la lista `citizenship` del último `profile`. `orden` es la posición de la nacionalidad en esa lista y `country` el nombre del país. Se descartan los países nulos.

Controles: cuántos jugadores tienen 1, 2, etc. nacionalidades, y cuántos jugadores de `silver_players` quedaron sin ninguna.

In [0]:
# EN LIMPIO: silver_player_citizenship (escribe workspace.inferiores.silver_player_citizenship)
q = """
WITH ult AS (
  SELECT player_id, body,
         ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY fecha DESC) AS rn
  FROM workspace.inferiores.bronze_player_raw
  WHERE llamada = 'profile' AND estado = 'hecho'
),
exp AS (
  SELECT player_id,
         posexplode(from_json(get_json_object(body, '$.citizenship'), 'array<string>')) AS (pos, country)
  FROM ult WHERE rn = 1
)
SELECT player_id, pos + 1 AS orden, country
FROM exp
WHERE country IS NOT NULL
"""
(spark.sql(q).write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.inferiores.silver_player_citizenship"))

# Chequeo 1: filas, jugadores y cuántas nacionalidades tiene cada uno
display(spark.sql("""
SELECT n_nacionalidades, COUNT(*) AS jugadores
FROM (SELECT player_id, COUNT(*) AS n_nacionalidades
      FROM workspace.inferiores.silver_player_citizenship GROUP BY player_id)
GROUP BY n_nacionalidades ORDER BY n_nacionalidades"""))

# Chequeo 2: jugadores de silver_players que quedaron sin ninguna nacionalidad
display(spark.sql("""
SELECT COUNT(*) AS sin_nacionalidad
FROM workspace.inferiores.silver_players p
LEFT ANTI JOIN workspace.inferiores.silver_player_citizenship c ON p.player_id = c.player_id"""))

## silver_market_value_history
Una fila por jugador y punto de la serie `marketValueHistory` del último `market_value`.

Columnas: `orden`, `date`, `age`, `club_id`, `club_name`, `market_value` (entero) y `updated_at`.

Control: filas, jugadores, claves únicas (`player_id`, `orden`) y valores de mercado nulos.

In [0]:
# EN LIMPIO: silver_market_value_history (escribe workspace.inferiores.silver_market_value_history)
q = """
WITH ult AS (
  SELECT player_id, body,
         ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY fecha DESC) AS rn
  FROM workspace.inferiores.bronze_player_raw
  WHERE llamada = 'market_value' AND estado = 'hecho'
),
base AS (
  SELECT player_id,
         to_timestamp(get_json_object(body, '$.updatedAt')) AS updated_at,
         from_json(get_json_object(body, '$.marketValueHistory'),
           'array<struct<age:int,date:string,clubId:string,clubName:string,marketValue:bigint>>') AS hist
  FROM ult WHERE rn = 1
)
SELECT player_id,
       pos + 1                AS orden,
       to_date(h.date)        AS date,
       h.age                  AS age,
       h.clubId               AS club_id,
       h.clubName             AS club_name,
       h.marketValue          AS market_value,
       updated_at
FROM (SELECT player_id, updated_at, posexplode(hist) AS (pos, h) FROM base)
"""
(spark.sql(q).write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.inferiores.silver_market_value_history"))

display(spark.sql("""
SELECT COUNT(*) AS filas,
       COUNT(DISTINCT player_id) AS jugadores,
       COUNT(DISTINCT player_id, orden) AS claves_unicas,
       SUM(CASE WHEN market_value IS NULL THEN 1 ELSE 0 END) AS valor_nulo
FROM workspace.inferiores.silver_market_value_history"""))

## silver_national_team_career
Una fila por jugador y selección, a partir del último `national_career`.

- Se descartan los duplicados exactos: filas del mismo jugador con los mismos valores en todos los campos. Se conserva la primera según su posición en la respuesta.
- `categoria` se calcula con el nombre de la selección:

| Regla sobre `team_name` | `categoria` |
|---|---|
| Termina en ` U` seguido de dos dígitos | `juvenil` |
| Contiene `Olympi` | `olimpica` |
| Termina en ` B` | `b` |
| Ninguna de las anteriores | `mayor` |

- `edad_max` es el número de las selecciones `U<edad>` (por ejemplo `U20` -> `20`). Es nulo en las demás.

Columnas restantes: `team_id`, `team_name`, `appearances`, `goals`, `shirt_number`, `is_captain`, `debut`, `last_match`, `status` y `updated_at`.

Controles: filas, jugadores, claves (`player_id`, `orden`) y (`player_id`, `team_id`), y filas por `categoria`.

In [0]:
# EN LIMPIO: silver_national_team_career (v2: sin duplicados exactos + categoria 'b')
q = """
WITH ult AS (
  SELECT player_id, body,
         ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY fecha DESC) AS rn
  FROM workspace.inferiores.bronze_player_raw
  WHERE llamada = 'national_career' AND estado = 'hecho'
),
base AS (
  SELECT player_id,
         to_timestamp(get_json_object(body, '$.updatedAt')) AS updated_at,
         from_json(get_json_object(body, '$.nationalTeams'),
           'array<struct<id:string,name:string,appearances:int,goals:int,shirtNumber:int,isCaptain:boolean,debut:string,lastMatch:string,status:string>>') AS eq
  FROM ult WHERE rn = 1
),
exp AS (
  SELECT player_id, updated_at, posexplode(eq) AS (pos, e) FROM base
),
dedup AS (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY player_id, e.id, e.name, e.appearances, e.goals, e.shirtNumber,
                        e.isCaptain, e.debut, e.lastMatch, e.status
           ORDER BY pos) AS dup
  FROM exp
)
SELECT player_id,
       ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY pos) AS orden,
       e.id                            AS team_id,
       e.name                          AS team_name,
       CASE WHEN e.name RLIKE ' U[0-9]{2}$' THEN 'juvenil'
            WHEN e.name RLIKE 'Olympi'      THEN 'olimpica'
            WHEN e.name RLIKE ' B$'         THEN 'b'
            ELSE 'mayor' END           AS categoria,
       try_cast(nullif(regexp_extract(e.name, ' U([0-9]{2})$', 1), '') AS INT) AS edad_max,
       e.appearances                   AS appearances,
       e.goals                         AS goals,
       e.shirtNumber                   AS shirt_number,
       e.isCaptain                     AS is_captain,
       to_date(e.debut)                AS debut,
       to_date(e.lastMatch)            AS last_match,
       e.status                        AS status,
       updated_at
FROM dedup WHERE dup = 1
"""
(spark.sql(q).write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.inferiores.silver_national_team_career"))

display(spark.sql("""
SELECT COUNT(*) AS filas, COUNT(DISTINCT player_id) AS jugadores,
       COUNT(DISTINCT player_id, orden)   AS claves_orden,
       COUNT(DISTINCT player_id, team_id) AS claves_equipo
FROM workspace.inferiores.silver_national_team_career"""))

display(spark.sql("""
SELECT categoria, COUNT(*) AS filas
FROM workspace.inferiores.silver_national_team_career
GROUP BY categoria ORDER BY filas DESC"""))

## silver_achievements
Una fila por jugador, título y detalle, a partir del último `achievements`. Cada título (`title`) tiene una lista de detalles; cada detalle es una temporada, un club y una competición.

Columnas: `orden_titulo`, `orden_detalle`, `title`, `season_id`, `season_name`, `club_id`, `club_name`, `competition_id`, `competition_name` y `updated_at`. Los textos se recortan con `trim`.

Control: filas, jugadores, claves únicas (`player_id`, `orden_titulo`, `orden_detalle`), detalles sin club y detalles sin competición.

In [0]:
# EN LIMPIO: silver_achievements (escribe workspace.inferiores.silver_achievements)
q = """
WITH ult AS (
  SELECT player_id, body,
         ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY fecha DESC) AS rn
  FROM workspace.inferiores.bronze_player_raw
  WHERE llamada = 'achievements' AND estado = 'hecho'
),
base AS (
  SELECT player_id,
         to_timestamp(get_json_object(body, '$.updatedAt')) AS updated_at,
         from_json(get_json_object(body, '$.achievements'),
           'array<struct<title:string,count:int,details:array<struct<season:struct<id:string,name:string>,club:struct<id:string,name:string>,competition:struct<id:string,name:string>>>>>') AS logros
  FROM ult WHERE rn = 1
),
titulos AS (
  SELECT player_id, updated_at, posexplode(logros) AS (pt, l) FROM base
)
SELECT player_id,
       pt + 1                          AS orden_titulo,
       pd + 1                          AS orden_detalle,
       trim(l.title)                   AS title,
       d.season.id                     AS season_id,
       trim(d.season.name)             AS season_name,
       d.club.id                       AS club_id,
       trim(d.club.name)               AS club_name,
       d.competition.id                AS competition_id,
       trim(d.competition.name)        AS competition_name,
       updated_at
FROM titulos
LATERAL VIEW posexplode(l.details) x AS pd, d
"""
(spark.sql(q).write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.inferiores.silver_achievements"))

display(spark.sql("""
SELECT COUNT(*) AS filas,
       COUNT(DISTINCT player_id) AS jugadores,
       COUNT(DISTINCT player_id, orden_titulo, orden_detalle) AS claves_unicas,
       SUM(CASE WHEN club_id IS NULL THEN 1 ELSE 0 END) AS sin_club,
       SUM(CASE WHEN competition_id IS NULL THEN 1 ELSE 0 END) AS sin_competicion
FROM workspace.inferiores.silver_achievements"""))

## silver_player_stats
Una fila por jugador y entrada de estadísticas, a partir del último `stats`.

Columnas: `orden`, `competition_id`, `competition_name`, `season_id`, `club_id`, `appearances`, `goals`, `assists`, `yellow_cards`, `second_yellow_cards`, `red_cards`, `minutes_played`, `goals_conceded`, `clean_sheets` y `updated_at`.

Control: filas, jugadores, claves (`player_id`, `orden`), claves naturales (`player_id`, `competition_id`, `season_id`, `club_id`) y filas con 0 partidos.

In [0]:
# EN LIMPIO: silver_player_stats (escribe workspace.inferiores.silver_player_stats)
q = """
WITH ult AS (
  SELECT player_id, body,
         ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY fecha DESC) AS rn
  FROM workspace.inferiores.bronze_player_raw
  WHERE llamada = 'stats' AND estado = 'hecho'
),
base AS (
  SELECT player_id,
         to_timestamp(get_json_object(body, '$.updatedAt')) AS updated_at,
         from_json(get_json_object(body, '$.stats'),
           'array<struct<competitionId:string,competitionName:string,seasonId:string,clubId:string,appearances:int,goals:int,assists:int,yellowCards:int,secondYellowCards:int,redCards:int,minutesPlayed:int,goalsConceded:int,cleanSheets:int>>') AS st
  FROM ult WHERE rn = 1
)
SELECT player_id,
       pos + 1                          AS orden,
       s.competitionId                  AS competition_id,
       trim(s.competitionName)          AS competition_name,
       s.seasonId                       AS season_id,
       s.clubId                         AS club_id,
       s.appearances                    AS appearances,
       s.goals                          AS goals,
       s.assists                        AS assists,
       s.yellowCards                    AS yellow_cards,
       s.secondYellowCards              AS second_yellow_cards,
       s.redCards                       AS red_cards,
       s.minutesPlayed                  AS minutes_played,
       s.goalsConceded                  AS goals_conceded,
       s.cleanSheets                    AS clean_sheets,
       updated_at
FROM (SELECT player_id, updated_at, posexplode(st) AS (pos, s) FROM base)
"""
(spark.sql(q).write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.inferiores.silver_player_stats"))

display(spark.sql("""
SELECT COUNT(*) AS filas,
       COUNT(DISTINCT player_id) AS jugadores,
       COUNT(DISTINCT player_id, orden) AS claves_orden,
       COUNT(DISTINCT player_id, competition_id, season_id, club_id) AS claves_naturales,
       SUM(CASE WHEN appearances = 0 THEN 1 ELSE 0 END) AS partidos_cero
FROM workspace.inferiores.silver_player_stats"""))

## silver_transfers
Una fila por jugador y traspaso, a partir del último `transfers`. La celda declara el esquema del JSON (`CLUB` y `SCHEMA`) y lo usa para descomponerlo.

- `season` es el texto de la temporada (por ejemplo `83/84`). `season_start_year` es el año en que empieza: con el formato `YY/YY`, un `YY` de 80 o más se toma como 19YY y uno menor como 20YY. Es nulo si la temporada no tiene ese formato.
- `from_*` y `to_*` describen el club de origen y el de destino: id, nombre, id y nombre del país, e id y nombre de la liga.
- Otras columnas: `orden`, `transfer_id`, `date`, `transfer_type`, `upcoming`, `age`, `market_value`, `fee`, `contract_until`, `remaining_contract_days` y `updated_at`.

Control: filas, jugadores, claves únicas (`player_id`, `transfer_id`), fechas nulas, filas sin año de temporada, traspasos futuros (`upcoming`) y traspasos con `fee` igual a 0.

In [0]:
# EN LIMPIO: silver_transfers
CLUB = "struct<id:string,name:string,countryId:string,countryName:string,league:struct<id:string,name:string>>"
SCHEMA = ("array<struct<id:string,clubFrom:" + CLUB + ",clubTo:" + CLUB +
          ",date:string,upcoming:boolean,transferType:string,season:string,marketValue:bigint,"
          "fee:bigint,age:int,contractUntil:string,remainingContractDays:int>>")

spark.sql(f"""
WITH ult AS (
  SELECT player_id, body,
         ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY fecha DESC) AS rn
  FROM workspace.inferiores.bronze_player_raw
  WHERE llamada = 'transfers' AND estado = 'hecho'
),
base AS (
  SELECT player_id,
         to_timestamp(get_json_object(body, '$.updatedAt')) AS updated_at,
         from_json(get_json_object(body, '$.transfers'), '{SCHEMA}') AS tr
  FROM ult WHERE rn = 1
),
t AS (
  SELECT player_id, updated_at, posexplode(tr) AS (pos, t) FROM base
)
SELECT
  player_id,
  pos + 1                                   AS orden,
  t.id                                      AS transfer_id,
  to_date(t.date)                           AS date,
  trim(t.season)                            AS season,
  CASE WHEN trim(t.season) RLIKE '^[0-9]{{2}}/[0-9]{{2}}$'
       THEN CASE WHEN CAST(substr(trim(t.season), 1, 2) AS INT) >= 80
                 THEN 1900 + CAST(substr(trim(t.season), 1, 2) AS INT)
                 ELSE 2000 + CAST(substr(trim(t.season), 1, 2) AS INT) END
  END                                       AS season_start_year,
  trim(t.transferType)                      AS transfer_type,
  t.upcoming,
  t.age,
  t.marketValue                             AS market_value,
  t.fee,
  to_date(t.contractUntil)                  AS contract_until,
  t.remainingContractDays                   AS remaining_contract_days,
  t.clubFrom.id                             AS from_club_id,
  trim(t.clubFrom.name)                     AS from_club_name,
  t.clubFrom.countryId                      AS from_country_id,
  trim(t.clubFrom.countryName)              AS from_country_name,
  t.clubFrom.league.id                      AS from_league_id,
  trim(t.clubFrom.league.name)              AS from_league_name,
  t.clubTo.id                               AS to_club_id,
  trim(t.clubTo.name)                       AS to_club_name,
  t.clubTo.countryId                        AS to_country_id,
  trim(t.clubTo.countryName)                AS to_country_name,
  t.clubTo.league.id                        AS to_league_id,
  trim(t.clubTo.league.name)                AS to_league_name,
  updated_at
FROM t
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.inferiores.silver_transfers")

# Verificación
display(spark.sql("""
SELECT COUNT(*)                                   AS filas,
       COUNT(DISTINCT player_id)                  AS jugadores,
       COUNT(DISTINCT player_id, transfer_id)     AS claves_unicas,
       SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END)              AS fecha_nula,
       SUM(CASE WHEN season_start_year IS NULL THEN 1 ELSE 0 END) AS sin_anio_temporada,
       SUM(CASE WHEN upcoming THEN 1 ELSE 0 END)  AS upcoming_true,
       SUM(CASE WHEN fee = 0 THEN 1 ELSE 0 END)   AS fee_cero
FROM workspace.inferiores.silver_transfers"""))

## silver_youth_clubs
Una fila por jugador y club juvenil, a partir de la lista `youthClubs` del último `transfers`. Columnas: `orden`, `club_name` (recortado) y `updated_at`. Se descartan los nombres vacíos.

Controles: filas, jugadores, claves únicas (`player_id`, `orden`), clubes distintos y nombres repetidos por jugador.

In [0]:
# EN LIMPIO: silver_youth_clubs
spark.sql("""
WITH ult AS (
  SELECT player_id, body,
         ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY fecha DESC) AS rn
  FROM workspace.inferiores.bronze_player_raw
  WHERE llamada = 'transfers' AND estado = 'hecho'
),
base AS (
  SELECT player_id,
         to_timestamp(get_json_object(body, '$.updatedAt')) AS updated_at,
         from_json(get_json_object(body, '$.youthClubs'), 'array<string>') AS yc
  FROM ult WHERE rn = 1
),
t AS (
  SELECT player_id, updated_at, posexplode(yc) AS (pos, nombre) FROM base
)
SELECT player_id,
       pos + 1       AS orden,
       trim(nombre)  AS club_name,
       updated_at
FROM t
WHERE trim(nombre) <> ''
""").write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.inferiores.silver_youth_clubs")

# Verificación
display(spark.sql("""
SELECT COUNT(*)                              AS filas,
       COUNT(DISTINCT player_id)             AS jugadores,
       COUNT(DISTINCT player_id, orden)      AS claves_unicas,
       COUNT(DISTINCT club_name)             AS clubes_distintos,
       COUNT(*) - COUNT(DISTINCT player_id, club_name) AS nombres_repetidos_por_jugador
FROM workspace.inferiores.silver_youth_clubs"""))